### Part 1 - ai_tasks x topics

In [1]:
import json
from tqdm import tqdm
import random

In [2]:
def is_word_in_end_of_prompt(prompt, n, word):
    """
    Checks if 'word' is present in the last n words of 'sentence'.
    
    :param sentence: A string representing the sentence
    :param n: An integer specifying how many words from the end to consider
    :param word: The target word to search for
    :return: True if 'word' is in the last n words of the sentence, otherwise False
    """
    # Split the sentence into a list of words
    words = prompt.lower().split()
    
    # Determine how many words to check from the end (handle edge cases)
    last_n_words = words[-n:] if n <= len(words) else words
    
    # Check if the given word is present in the last n words
    return word.lower() in last_n_words

    
def answer_style_specification(prompt, answer_styles_data):
    random_number = random.randint(1, 100)
    if is_word_in_end_of_prompt(prompt, 15, "Please"): # If there is already a style specification in the prompt -> do not overwrite it / give two confusing style specifications
        return "None because already in prompt", None
    elif random_number > 45: # Adjusted to have ~50% of prompts with style specifications
        return "Randomly None", None
    elif is_word_in_end_of_prompt(prompt, len(prompt), "Emergency") and random_number > 25: # Give clear style specifications in the case of an emergency
        return "Emergency", "Highlight the urgent steps to take."
    else:
        # Set weights for titles (must correspond to the order in the options JSON file)
        words = ["translate", "translation", "english"]
        if any(word in prompt.lower() for word in words):
            weights = [3, 3, 3, 0, 1, 1]  # do not want the output given in a specified language, since it is already specified in the prompt
        else:
            weights = [3, 3, 3, 3, 1, 1]
        style_categories = [item['title'] for item in answer_styles_data]
        style_category = random.choices(style_categories, weights=weights, k=1)[0]
        style_specifications = next(item['options'] for item in answer_styles_data if item['title'] == style_category)
        style_specification = random.choice(style_specifications)

        return style_category, style_specification
    
def get_profession(id): # necessary because silly mistake -> forgot to add profession in id of batched prompt
    dict = {0: "Physician",
            1: "Nurse",
            2: "Physician Assistant",
            3: "Pharmacist",
            4: "Physical Therapist",
            5: "Occupational Therapist",
            6: "Speech-Language Pathologist",
            7: "Radiologic Technologist",
            8: "Medical Laboratory Scientist",
            9: "Dietitian",
            10: "Respiratory Therapist",
            11: "Paramedic",
            12: "Chiropractor",
            13: "Podiatrist",
            14: "Optometrist",
            15: "Audiologist",
            16: "Occupational Health and Safety Specialist",
            17: "Medical Social Worker",
            18: "Genetic Counselor",
            19: "Orthotist and Prosthetist",
            20: "Clinical Psychologist"
            }
    return dict.get(id % 21)

# Function to parse the string and create a JSON object
def parse_gpt_response(string, id, task, topic, answer_styles_data):
    """
    Helper function fur parse_results()
    Parses the gpt responses from string to dict
    """
# Create a dictionary to hold the data in the desired format
# Only successfull if all that is expected is present

    try:
        # Attempt to load the JSON string
        # Decode any UTF-8 character codes in the input string
        decoded_string = (string.encode().decode('unicode_escape')).encode('latin1').decode('utf-8')
        style_category, answer_style = answer_style_specification(prompt=decoded_string, answer_styles_data=answer_styles_data)
        
        if answer_style != None:
            prompt_parts = []
            prompt_parts.append(decoded_string)
            prompt_parts.append(answer_style)
            prompt = ' '.join(prompt_parts)
        else:
            prompt = decoded_string

        data_dict = {}
        data_dict["id"] = id
        data_dict["prompt"] = prompt
        data_dict["context"] = {"task": task, "topic": topic, "profession": get_profession(id), "answer_style": style_category}
        return data_dict, style_category

    except json.JSONDecodeError as e:
        return None    

In [3]:
path_to_results = "../results/gpt_results.jsonl"
path_to_answer_styles = "../resources/answer_styles.json"
with open(path_to_answer_styles, 'r') as f:
    answer_styles_data = json.load(f)
output_path = "../results/parsed_prompts_tasks_x_topics_x_answerstyles.json"


# TODO: do both in the same for loop, no need to store contents
# Extract content from json
print("Parsing GPT responses...")
medical_prompts = []
questions_failed_to_parse = []
nbr_of_prompts_without_answer_styles = 0
nbr_of_prompts_initially_containing_answer_styles = 0
with open(path_to_results, 'r') as file:
    for line in tqdm(file):
        try:
            data = json.loads(line)  # Parse each line as JSON
            response_content = data.get("response", {}).get("body", {}).get("choices", [])[0].get("message", {}).get("content", None)
            id = data.get("custom_id") # format "0-0" "task_id - subtopic_id"
            if response_content and id:
                parsed_response, style_category = parse_gpt_response(string=response_content, id = int(id.split("-")[0]),
                                                                   task=str(id.split("-")[1]), topic=str(id.split("-")[2]),
                                                                   answer_styles_data=answer_styles_data)
                if bool(parsed_response):
                    medical_prompts.append(parsed_response)
                else:
                    questions_failed_to_parse.append(id)
                if style_category == "Randomly None":
                    nbr_of_prompts_without_answer_styles += 1
                if style_category == "None because already in prompt":
                    nbr_of_prompts_initially_containing_answer_styles += 1
        except json.JSONDecodeError as e:
            questions_failed_to_parse.append(id)
print("Parsing completed")

# Save to JSON file
with open(output_path, 'w') as json_file:
    json.dump(medical_prompts, json_file, indent=4)

# Output the parsed data (for verification)
print(f"Medical prompts have been saved to {output_path}")
print("See an example below:")
print(json.dumps(parsed_response, indent=4))
print(f"Failed to parse {len(questions_failed_to_parse)} questions:")
print(questions_failed_to_parse)
print(f"{nbr_of_prompts_without_answer_styles} have no style specifications")
print(f"{nbr_of_prompts_initially_containing_answer_styles} already had style specifications")

Parsing GPT responses...


0it [00:00, ?it/s]

13377it [00:00, 14242.22it/s]


Parsing completed
Medical prompts have been saved to ../results/parsed_prompts_tasks_x_topics_x_answerstyles.json
See an example below:
{
    "id": 13376,
    "prompt": "I am a clinical psychologist reviewing the compliance of a mental health clinic's preventive care program aimed at reducing the incidence of anxiety in adolescents. The program involves regular screening using standardized psychological assessment tools and providing early intervention. Considering the current healthcare regulations and policies, what key compliance requirements should I ensure the program meets to uphold privacy and efficacy standards? Additionally, how can I integrate evidence-based practices while adhering to policy guidelines?",
    "context": {
        "task": "Health Policy Guidance",
        "topic": "Preventive Medicine",
        "profession": "Clinical Psychologist",
        "answer_style": "Randomly None"
    }
}
Failed to parse 0 questions:
[]
6528 have no style specifications
1466 already h

In [5]:
# Randomly sample and display some entries to check if they are valid

def sample_random_entries(json_file_path: str, n: int):
    """
    Reads a list of entries from a .json file, randomly samples n entries, 
    and prints them to the console.
    
    :param json_file_path: Path to the .json file.
    :param n: Number of entries to sample and print.
    """
    # Load data from the JSON file
    with open(json_file_path, 'r', encoding='utf-8') as file:
        data = json.load(file)
    
    # Ensure n does not exceed the total number of entries
    n = min(n, len(data))
    
    # Randomly sample n entries
    sampled_entries = random.sample(data, n)
    
    return sampled_entries
sample_random_entries("../results/parsed_prompts_tasks_x_topics_x_answerstyles.json", 10)


[{'id': 7855,
  'prompt': 'I am a nurse working in a hematology clinic in Chicago. A 67-year-old male patient with anemia of chronic disease presents after his recent blood work. His lab results show a hemoglobin level of 9.5 g/dL, MCV of 86 fL, and serum ferritin of 500 ng/mL. How should these results be interpreted in the context of his condition, and what further investigations or management strategies would be recommended to optimize his blood levels? Please provide the response as a paragraph.',
  'context': {'task': 'Lab Test Interpretation',
   'topic': 'Hematology',
   'profession': 'Nurse',
   'answer_style': 'Formatting'}},
 {'id': 11143,
  'prompt': 'I am a podiatrist seeking educational handouts on diabetic foot care for a Spanish-speaking patient with limited English proficiency. The following information needs translation: "Proper foot care is critical for individuals with diabetes to prevent complications. Inspect your feet daily for any cuts, blisters, redness, or swell